# Fragrantica Accord Analysis

## Objective

Analyze the prevalence, distribution, and rating patterns of fragrance accords in the cleaned Fragrantica reference dataset. The analysis examines the most common accords, differences across gender categories, and rating variations relative to gender-specific baselines.

The objective is to identify meaningful patterns in accord prevalence and fragrance ratings and prepare structured analytical results for visualization and further exploration.

## Data Source

- **Source table:** `main.fragrance_analytics.fragrantica_perfumes`
- **Format:** Delta table
- **Platform:** Databricks

The source table contains cleaned fragrance records, including fragrance names, standardized gender classifications (`women`, `men`, and `unisex`), ratings, main accords, perfumers, descriptions, and source URLs.

## Workflow

1. Inspect the structure and format of the `main_accords` column.
2. Parse accord lists stored as strings into individual accord values.
3. Explode the lists to create a fragrance–accord analytical dataset.
4. Validate the parsed data and identify missing or invalid accord values.
5. Analyze overall accord frequency and prevalence.
6. Compare accord prevalence across gender categories.
7. Calculate average and median ratings for accords within each gender category.
8. Compare accord ratings against gender-specific rating baselines.
9. Identify the highest-rated accords and the largest positive and negative rating differences, applying a minimum sample-size threshold.
10. Prepare analytical results for visualization and interpretation.

## Expected Outputs

- A fragrance–accord dataset containing individual fragrance–accord combinations.
- Overall accord frequency and prevalence metrics.
- Gender-specific accord prevalence comparisons.
- Accord-level rating statistics, including average and median ratings.
- Comparisons of accord ratings against gender-specific baselines.
- Ranked results highlighting highly rated accords and accords with the largest rating differences.

These outputs provide a foundation for fragrance analytics, data visualization, and potential natural-language exploration through Databricks AI/BI Genie.

**Interpretation note:** Rating differences and accord prevalence represent observed associations in the dataset. They do not establish causation or directly measure consumer preferences.

## Technologies

Databricks · Python · PySpark · SQL · Delta Lake

In [0]:

from pyspark.sql import functions as F

# Load the cleaned Fragrantica reference table
df_perfumes = spark.table(
    "main.fragrance_analytics.fragrantica_perfumes"
)

# Inspect the schema
df_perfumes.printSchema()

# Check total records and missing accord values
df_perfumes.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.when(
            F.col("main_accords").isNull()
            | (F.trim(F.col("main_accords")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_accords")
).show()

# Display sample accord values
df_perfumes.select(
    "name",
    "gender",
    "main_accords"
).show(20, truncate=False)


In [0]:

from pyspark.sql import functions as F

# Parse the string representation of accord lists into arrays
df_accords = df_perfumes.withColumn(
    "accords",
    F.from_json(
        F.regexp_replace(
            F.regexp_replace(
                F.col("main_accords"),
                "'", '"'
            ),
            r"\bNone\b", "null"
        ),
        "array<string>"
    )
)

# Explode the array: one row per fragrance–accord combination
df_fragrance_accords = (
    df_accords
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        F.explode("accords").alias("accord")
    )
    .withColumn("accord", F.trim(F.lower(F.col("accord"))))
    .filter(F.col("accord").isNotNull() & (F.col("accord") != ""))
)

# Validate the result
print("Original fragrance records:", df_perfumes.count())
print("Fragrance–accord rows:", df_fragrance_accords.count())
print(
    "Distinct accords:",
    df_fragrance_accords.select("accord").distinct().count()
)

display(df_fragrance_accords.limit(20))


In [0]:

from pyspark.sql import functions as F

# 1. Check how many records produced a valid, non-empty accord array
df_accords.select(
    F.count("*").alias("total_fragrances"),
    F.sum(
        F.when(F.col("accords").isNull(), 1).otherwise(0)
    ).alias("null_accord_arrays"),
    F.sum(
        F.when(
            F.col("accords").isNotNull()
            & (F.size("accords") == 0),
            1
        ).otherwise(0)
    ).alias("empty_accord_arrays")
).show()

# 2. Check whether any fragrance has duplicate accord entries
duplicate_accords = (
    df_fragrance_accords
    .groupBy("id", "accord")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate fragrance–accord combinations:",
    duplicate_accords.count()
)

# 3. Summarize the number of accords per fragrance
accords_per_fragrance = (
    df_fragrance_accords
    .groupBy("id")
    .agg(F.countDistinct("accord").alias("accord_count"))
)

accords_per_fragrance.select(
    F.min("accord_count").alias("min_accords"),
    F.avg("accord_count").alias("avg_accords"),
    F.max("accord_count").alias("max_accords")
).show()


In [0]:

from pyspark.sql import functions as F

# Count distinct accord labels at each stage
print(
    "Distinct non-empty accord strings in the original arrays:",
    df_accords
    .select(F.explode("accords").alias("accord"))
    .filter(F.col("accord").isNotNull() & (F.trim("accord") != ""))
    .select("accord")
    .distinct()
    .count()
)

print(
    "Distinct normalized accords after exploding:",
    df_fragrance_accords.select("accord").distinct().count()
)

# Show all distinct accord labels from the exploded dataset
display(
    df_fragrance_accords
    .select("accord")
    .distinct()
    .orderBy("accord")
)


In [0]:

from pyspark.sql import functions as F

# Count fragrances containing each accord
accord_prevalence = (
    df_fragrance_accords
    .groupBy("accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
)

# Denominator: fragrances with at least one accord
fragrances_with_accords = (
    df_fragrance_accords
    .select("id")
    .distinct()
    .count()
)

# Calculate prevalence and rank accords
accord_prevalence = (
    accord_prevalence
    .withColumn(
        "prevalence_pct",
        F.round(
            F.col("fragrance_count") / F.lit(fragrances_with_accords) * 100,
            2
        )
    )
    .orderBy(F.desc("fragrance_count"), F.asc("accord"))
)

print("Fragrances with at least one accord:", fragrances_with_accords)

display(accord_prevalence)


In [0]:
gender_counts = (
    df_fragrance_accords
    .select("id", "gender")
    .distinct()
    .groupBy("gender")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy(F.desc("fragrance_count"))
)

display(gender_counts)

# Count distinct fragrances in each gender group
gender_denominators = (
    df_fragrance_accords
    .select("id", "gender")
    .distinct()
    .groupBy("gender")
    .agg(
        F.countDistinct("id").alias("gender_fragrance_count")
    )
)

# Count fragrances containing each accord within each gender group
accords_by_gender = (
    df_fragrance_accords
    .groupBy("gender", "accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .join(gender_denominators, on="gender", how="inner")
    .withColumn(
        "prevalence_pct",
        F.round(
            F.col("fragrance_count")
            / F.col("gender_fragrance_count") * 100,
            2
        )
    )
    .orderBy("gender", F.desc("prevalence_pct"), F.asc("accord"))
)

display(accords_by_gender)

In [0]:

# Compare accord prevalence across gender groups
accord_gender_comparison = (
    accords_by_gender
    .groupBy("accord")
    .agg(
        F.coalesce(
            F.max(
                F.when(
                    F.col("gender") == "men",
                    F.col("prevalence_pct")
                )
            ),
            F.lit(0.0)
        ).alias("men_pct"),

        F.coalesce(
            F.max(
                F.when(
                    F.col("gender") == "women",
                    F.col("prevalence_pct")
                )
            ),
            F.lit(0.0)
        ).alias("women_pct"),

        F.coalesce(
            F.max(
                F.when(
                    F.col("gender") == "unisex",
                    F.col("prevalence_pct")
                )
            ),
            F.lit(0.0)
        ).alias("unisex_pct")
    )
    .withColumn(
        "max_prevalence_pct",
        F.greatest("men_pct", "women_pct", "unisex_pct")
    )
    .withColumn(
        "min_prevalence_pct",
        F.least("men_pct", "women_pct", "unisex_pct")
    )
    .withColumn(
        "gap_pp",
        F.round(
            F.col("max_prevalence_pct")
            - F.col("min_prevalence_pct"),
            2
        )
    )
    .orderBy(F.desc("gap_pp"), F.asc("accord"))
)

display(accord_gender_comparison)


In [0]:

# Calculate overall prevalence for each accord
total_fragrances_with_accords = (
    df_fragrance_accords
    .select("id")
    .distinct()
    .count()
)

overall_accord_prevalence = (
    df_fragrance_accords
    .groupBy("accord")
    .agg(
        F.countDistinct("id").alias("overall_fragrance_count")
    )
    .withColumn(
        "overall_prevalence_pct",
        F.round(
            F.col("overall_fragrance_count")
            / F.lit(total_fragrances_with_accords) * 100,
            2
        )
    )
)

# Focus on commonly occurring accords with differences across groups
common_accord_comparison = (
    accord_gender_comparison
    .join(overall_accord_prevalence, on="accord", how="left")
    .filter(F.col("overall_prevalence_pct") >= 10)
    .select(
        "accord",
        "overall_fragrance_count",
        "overall_prevalence_pct",
        "men_pct",
        "women_pct",
        "unisex_pct",
        "gap_pp"
    )
    .orderBy(F.desc("gap_pp"))
)

display(common_accord_comparison)


In [0]:

# One record per fragrance, including its rating and gender
fragrance_ratings = (
    df_perfumes
    .select(
        "id",
        "gender",
        "rating_value",
        "rating_count"
    )
    .dropDuplicates(["id"])
)

# Aggregate ratings for each accord and gender group
accord_rating_by_gender = (
    df_fragrance_accords
    .select("id", "gender", "accord")
    .distinct()
    .join(
        fragrance_ratings.select(
            "id", "rating_value", "rating_count"
        ),
        on="id",
        how="left"
    )
    .groupBy("gender", "accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.count("rating_value").alias("rated_fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("avg_rating"),
        F.percentile_approx(
            "rating_value", 0.5, 10000
        ).alias("median_rating"),
        F.sum("rating_count").alias("total_rating_count"),
        F.round(F.avg("rating_count"), 1).alias("avg_rating_count")
    )
    .orderBy("gender", F.desc("avg_rating"))
)

display(accord_rating_by_gender)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Window

# Minimum sample size for reliable comparison
MIN_RATED_FRAGRANCES = 100

# 1. Calculate the overall rating baseline for each gender
gender_rating_baseline = (
    fragrance_ratings
    .groupBy("gender")
    .agg(
        F.count("rating_value").alias("baseline_rated_count"),
        F.round(F.avg("rating_value"), 3).alias("gender_avg_rating"),
        F.round(
            F.percentile_approx("rating_value", 0.5, 10000), 3
        ).alias("gender_median_rating")
    )
)

# 2. Prepare one shared dataset of accord ratings and gender baselines
accord_rating_analysis = (
    accord_rating_by_gender
    .filter(F.col("rated_fragrance_count") >= MIN_RATED_FRAGRANCES)
    .join(gender_rating_baseline, on="gender", how="left")
    .withColumn(
        "rating_difference",
        F.round(F.col("avg_rating") - F.col("gender_avg_rating"), 3)
    )
)

# 3. Rank accords by average rating within each gender
rating_window = (
    Window
    .partitionBy("gender")
    .orderBy(
        F.desc("avg_rating"),
        F.desc("median_rating"),
        F.desc("rated_fragrance_count")
    )
)

top_rated_accords = (
    accord_rating_analysis
    .withColumn("rank", F.row_number().over(rating_window))
    .filter(F.col("rank") <= 10)
    .select(
        "gender",
        "rank",
        "accord",
        "rated_fragrance_count",
        "avg_rating",
        "median_rating",
        "avg_rating_count"
    )
    .orderBy("gender", "rank")
)

# 4. Compare each accord's average rating with its gender baseline
accord_rating_vs_baseline = (
    accord_rating_analysis
    .select(
        "gender",
        "accord",
        "rated_fragrance_count",
        "avg_rating",
        "gender_avg_rating",
        "rating_difference",
        "median_rating",
        "gender_median_rating"
    )
    .orderBy("gender", F.desc("rating_difference"))
)

# 5. Display both analytical views
print("Top 10 highest-rated accords within each gender")
display(top_rated_accords)

print("Rating differences relative to each gender's average")
display(accord_rating_vs_baseline)


In [0]:

# Rank accords by rating difference within each gender
rating_difference_window = (
    Window
    .partitionBy("gender")
    .orderBy(F.desc("rating_difference"), F.asc("accord"))
)

rating_difference_ranked = (
    accord_rating_vs_baseline
    .withColumn(
        "positive_rank",
        F.row_number().over(rating_difference_window)
    )
)

# Top 10 accords above the gender baseline
top_above_baseline = (
    rating_difference_ranked
    .filter(F.col("positive_rank") <= 10)
    .withColumn("comparison", F.lit("Above baseline"))
)

# Bottom 10 accords below the gender baseline
negative_window = (
    Window
    .partitionBy("gender")
    .orderBy(F.asc("rating_difference"), F.asc("accord"))
)

bottom_below_baseline = (
    accord_rating_vs_baseline
    .withColumn(
        "negative_rank",
        F.row_number().over(negative_window)
    )
    .filter(F.col("negative_rank") <= 10)
    .withColumn("comparison", F.lit("Below baseline"))
)

# Combine both views
rating_extremes = (
    top_above_baseline
    .select(
        "gender", "comparison", "accord",
        "rated_fragrance_count", "avg_rating",
        "gender_avg_rating", "rating_difference"
    )
    .unionByName(
        bottom_below_baseline.select(
            "gender", "comparison", "accord",
            "rated_fragrance_count", "avg_rating",
            "gender_avg_rating", "rating_difference"
        )
    )
    .orderBy(
        "gender",
        "comparison",
        F.desc("rating_difference")
    )
)

display(rating_extremes)